# AksharDrishti — Kaggle Setup Notebook

**Notebook name:** `akshardrishti-train`  
**Accelerator:** GPU T4 x2 — set in *Settings → Accelerator*  
**Internet:** ON — set in *Settings → Internet*  
**Persistence:** Files only — set in *Settings → Persistence*  

Run cells top-to-bottom in order. Each cell prints its own PASS/FAIL so you can stop immediately if something is wrong rather than discovering the problem two hours into training.

**This notebook does NOT start any training.** It only sets up the environment.

---
## STEP 1 — GPU sanity check

In [ ]:
import subprocess, sys, os

print('=== Python ===')
print(sys.version)

print('\n=== nvidia-smi ===')
try:
    print(subprocess.check_output(['nvidia-smi'], stderr=subprocess.STDOUT).decode())
except Exception as e:
    print('ERROR: no GPU detected:', e)
    print('Go to Settings → Accelerator → GPU T4 x2, then re-run.')

print('\n=== PyTorch GPU check ===')
import torch
print('torch version  :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
print('Device count   :', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

assert torch.cuda.is_available(), 'STOP — no CUDA. Enable GPU T4 x2 in Settings.'

---
## STEP 2 — Clone repo and check out exact commit `594e7ff`

In [ ]:
REPO   = 'https://github.com/Ayush-04-spec/akshardrishti.git'
COMMIT = '594e7ff'
TARGET = '/kaggle/working/aksharDrishti'

import os, subprocess

def run(cmd, **kw):
    """Run a shell command, print output, raise on non-zero exit."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    result.check_returncode()
    return result.stdout.strip()

if not os.path.isdir(TARGET):
    print(f'Cloning {REPO} ...')
    run(f'git clone {REPO} {TARGET}')
else:
    print(f'Repo already exists at {TARGET}, skipping clone.')

print(f'\nChecking out commit {COMMIT} ...')
run(f'git -C {TARGET} checkout {COMMIT}')

actual = run(f'git -C {TARGET} rev-parse HEAD')
print(f'\nHEAD is now: {actual}')
assert actual.startswith(COMMIT), f'STOP — HEAD {actual!r} does not start with {COMMIT!r}'
print(f'PASS — commit matches {COMMIT}')

# Add repo to Python path for all subsequent cells
import sys
if TARGET not in sys.path:
    sys.path.insert(0, TARGET)
os.chdir(TARGET)
print(f'cwd → {os.getcwd()}')

---
## STEP 3 — Verify attached datasets

Before running this cell, make sure both datasets are attached as notebook inputs:
- **Add Data** (top-right panel) → search `akshardrishti-indicdlp-subset` → Add  
- **Add Data** → search `mozhi-dataset` → Add

In [ ]:
import os

INDICDLP_ROOT = '/kaggle/input/datasets/ayushshirsath03/akshardrishti-indicdlp-subset/indicdlp_subset'
MOZHI_ROOT    = '/kaggle/input/datasets/ayushshirsath03/mozhi-dataset'

# ── IndicDLP ────────────────────────────────────────────────────────────────
print('='*60)
print('IndicDLP subset:', INDICDLP_ROOT)
print('='*60)

assert os.path.isdir(INDICDLP_ROOT), (
    f'STOP — {INDICDLP_ROOT} not found.\n'
    'Add akshardrishti-indicdlp-subset as a notebook input (Add Data panel).'
)

indicdlp_top = sorted(os.listdir(INDICDLP_ROOT))
print('Top-level contents:')
for name in indicdlp_top:
    full = os.path.join(INDICDLP_ROOT, name)
    if os.path.isdir(full):
        n = sum(len(files) for _, _, files in os.walk(full))
        print(f'  {name}/  ({n:,} files)')
    else:
        print(f'  {name}  ({os.path.getsize(full):,} bytes)')

# Critical check: images/ and labels/ must exist and be non-empty
images_dir = os.path.join(INDICDLP_ROOT, 'images', 'train')
labels_dir = os.path.join(INDICDLP_ROOT, 'labels', 'train')

assert os.path.isdir(images_dir), (
    f'STOP — images/train/ missing from IndicDLP dataset.\n'
    f'The upload is incomplete (only metadata was uploaded).\n'
    f'Re-upload indicdlp_subset.zip via the Kaggle web interface.'
)
assert os.path.isdir(labels_dir), (
    f'STOP — labels/train/ missing from IndicDLP dataset.\n'
    f'Re-upload indicdlp_subset.zip via the Kaggle web interface.'
)

n_images = len(os.listdir(images_dir))
n_labels = len(os.listdir(labels_dir))
print(f'\nimages/train/: {n_images:,} files')
print(f'labels/train/: {n_labels:,} files')
assert n_images >= 9000, f'STOP — only {n_images} images, expected ~9,306. Incomplete upload.'
assert n_labels >= 9000, f'STOP — only {n_labels} labels, expected ~9,306. Incomplete upload.'
print(f'PASS — IndicDLP: {n_images:,} images, {n_labels:,} labels')

# ── Mozhi ───────────────────────────────────────────────────────────────────
print()
print('='*60)
print('Mozhi raw:', MOZHI_ROOT)
print('='*60)

assert os.path.isdir(MOZHI_ROOT), (
    f'STOP — {MOZHI_ROOT} not found.\n'
    'Add akshardrishti-mozhi-raw (mozhi-dataset) as a notebook input (Add Data panel).'
)

mozhi_top = sorted(os.listdir(MOZHI_ROOT))
print('Top-level contents:')
for name in mozhi_top:
    full = os.path.join(MOZHI_ROOT, name)
    if os.path.isdir(full):
        n = sum(len(files) for _, _, files in os.walk(full))
        print(f'  {name}/  ({n:,} files)')
    else:
        print(f'  {name}  ({os.path.getsize(full):,} bytes)')

# Critical check: hindi/ and marathi/ image folders must exist
for lang in ('hindi', 'marathi'):
    for split in ('train', 'val', 'test'):
        d = os.path.join(MOZHI_ROOT, lang, split, 'images')
        assert os.path.isdir(d), (
            f'STOP — {d} not found.\n'
            f'The mozhi_raw.zip upload may be incomplete or the ZIP used backslash paths.\n'
            f'Re-upload using the Python-generated mozhi_raw.zip (forward-slash paths).'
        )
        n = len(os.listdir(d))
        print(f'  {lang}/{split}/images/: {n:,} files')

# Check index CSVs are present
for csv_name in ('index_train.csv', 'index_val.csv', 'index_test.csv', 'charset_deva.txt'):
    path = os.path.join(MOZHI_ROOT, csv_name)
    assert os.path.isfile(path), f'STOP — {csv_name} missing from Mozhi dataset.'
    print(f'  {csv_name}: {os.path.getsize(path):,} bytes  ✓')

print(f'\nPASS — both datasets look complete.')

---
## STEP 4 — Install dependencies

Kaggle's base image already has `torch`, `torchvision`, `numpy`, `opencv`, and `pandas`.  
We record what's already there **before** installing so the diff is clear.

In [ ]:
import subprocess

PKGS_TO_CHECK = 'torch|torchvision|ultralytics|sahi|pydantic|transformers|datasets|huggingface'

def pip_grep(pattern):
    out = subprocess.run(
        f'pip list 2>/dev/null | grep -iE "{pattern}"',
        shell=True, capture_output=True, text=True
    ).stdout.strip()
    return out if out else '(none)'

print('=== Package versions BEFORE install ===')
print(pip_grep(PKGS_TO_CHECK))

In [ ]:
# Install ONLY what Kaggle's base image doesn't already provide.
# torch / torchvision / numpy / opencv are pre-installed — do NOT force-reinstall them.

print('Installing ultralytics, sahi, pydantic, transformers, datasets, jiwer ...')
!pip install -q \
    'ultralytics>=8.3.0' \
    'sahi>=0.11.15' \
    'pydantic>=2.6' \
    'transformers>=4.40' \
    'datasets>=2.18' \
    'huggingface-hub>=0.22' \
    'jiwer>=3.0' \
    'wandb>=0.16' \
    'pymupdf'

print('\nInstalling Tesseract system package + language packs ...')
!apt-get install -qq -y \
    tesseract-ocr \
    tesseract-ocr-hin \
    tesseract-ocr-mar \
    fonts-indic \
    fonts-noto-core \
    > /dev/null 2>&1

print('Installing pytesseract ...')
!pip install -q pytesseract

print('\nDone.')

In [ ]:
print('=== Package versions AFTER install ===')
print(pip_grep(PKGS_TO_CHECK))

# Hard assertions so training cells fail fast if a key package is missing
import importlib
required = {
    'ultralytics': 'ultralytics',
    'sahi':        'sahi',
    'pydantic':    'pydantic',
    'transformers':'transformers',
    'pytesseract': 'pytesseract',
}
all_ok = True
for display_name, module_name in required.items():
    try:
        mod = importlib.import_module(module_name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  {display_name:<14} {ver}  ✓')
    except ImportError:
        print(f'  {display_name:<14} MISSING  ✗')
        all_ok = False

assert all_ok, 'STOP — one or more required packages failed to install. See ✗ above.'
print('\nPASS — all required packages installed.')

In [ ]:
# Smoke-test: import the akshardrishti package itself
import sys, os
TARGET = '/kaggle/working/aksharDrishti'
if TARGET not in sys.path:
    sys.path.insert(0, TARGET)
os.chdir(TARGET)

import akshardrishti
print(f'akshardrishti version: {akshardrishti.__version__}')

from akshardrishti.config import Config, ClassMap
from akshardrishti.schema import Document, Page, Region, RegionType, BBox
from akshardrishti.recognize.crnn_model import CRNN, Charset

cfg = Config.load('configs/pipeline.yaml')
cm  = ClassMap.load('configs/class_map.yaml')
print(f'Config loaded  — hash: {cfg.hash()}')
print(f'ClassMap loaded — {len(cm.targets)} target classes: {cm.targets}')
print('\nPASS — package imports OK.')

---
## STEP 5 — Fix Mozhi CSV paths

The index CSVs contain absolute Windows paths from the machine where they were generated.  
This step rewrites them to `/kaggle/input/datasets/ayushshirsath03/mozhi-dataset/...` and saves the fixed  
versions to `/kaggle/working/mozhi_fixed/` (writable, unlike `/kaggle/input/`).

In [ ]:
!python /kaggle/working/aksharDrishti/scripts/fix_mozhi_paths_kaggle.py \
    --input-dir  /kaggle/input/datasets/ayushshirsath03/mozhi-dataset \
    --output-dir /kaggle/working/mozhi_fixed \
    --spot-check-n 5

In [ ]:
# Confirm the fixed CSVs exist and spot-check 5 rows resolve to real files
import os, csv, random

FIXED_DIR = '/kaggle/working/mozhi_fixed'

for csv_name in ('index_train.csv', 'index_val.csv', 'index_test.csv'):
    path = os.path.join(FIXED_DIR, csv_name)
    assert os.path.isfile(path), f'STOP — {path} not found. Did the fix script run successfully?'
    with open(path, encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f'{csv_name}: {len(rows):,} rows')

print()

# Extended spot-check: 5 random rows from train
train_csv = os.path.join(FIXED_DIR, 'index_train.csv')
with open(train_csv, encoding='utf-8') as f:
    all_rows = list(csv.DictReader(f))

rng = random.Random(99)
sample = rng.sample(all_rows, 5)

print('Spot-check (5 random rows from index_train.csv):')
all_exist = True
for row in sample:
    exists = os.path.exists(row['image_path'])
    all_exist = all_exist and exists
    mark = 'OK' if exists else 'MISSING'
    print(f'  [{mark}]  {row["language"]:<8}  {row["text"]:<20}  {row["image_path"]}')

assert all_exist, (
    'STOP — some image files are missing.\n'
    'The Mozhi dataset upload may be incomplete. '
    'Verify akshardrishti-mozhi-raw on Kaggle shows ~566 MB.'
)
print('\nPASS — all 5 sampled files exist on disk.')

---
## Setup complete — summary

All five steps passed. Record the key values below for the lab report / paper.

In [ ]:
import subprocess, torch, importlib, os

TARGET = '/kaggle/working/aksharDrishti'

commit = subprocess.run(
    f'git -C {TARGET} rev-parse HEAD',
    shell=True, capture_output=True, text=True
).stdout.strip()

def ver(mod_name):
    try:
        return importlib.import_module(mod_name).__version__
    except Exception:
        return 'n/a'

print('=' * 56)
print('  AksharDrishti — Setup Summary')
print('=' * 56)
print(f'  Commit          : {commit}')
print(f'  PyTorch         : {torch.__version__}')
print(f'  CUDA            : {torch.version.cuda}')
print(f'  GPU count       : {torch.cuda.device_count()}')
print(f'  ultralytics     : {ver("ultralytics")}')
print(f'  sahi            : {ver("sahi")}')
print(f'  pydantic        : {ver("pydantic")}')
print(f'  transformers    : {ver("transformers")}')
print(f'  pytesseract     : {ver("pytesseract")}')
print(f'  IndicDLP images : {len(os.listdir("/kaggle/input/datasets/ayushshirsath03/akshardrishti-indicdlp-subset/indicdlp_subset/images/train")):,}')
print(f'  IndicDLP labels : {len(os.listdir("/kaggle/input/datasets/ayushshirsath03/akshardrishti-indicdlp-subset/indicdlp_subset/labels/train")):,}')
print(f'  Mozhi fixed CSVs: /kaggle/working/mozhi_fixed/')
print('=' * 56)
print('  STATUS: READY FOR TRAINING')
print('=' * 56)